In [ ]:
# === ARRANQUE EN COLAB: arbol de carpetas de la sesion =====================
# Este cuaderno se escribio para correr desde la carpeta `notebook/` de su
# sesion, con ../data, ../figuras y ../resultados al lado. Colab arranca en
# /content y sin ese arbol, asi que aqui se recrea y nos situamos dentro: con
# eso, todas las rutas relativas del cuaderno funcionan igual que en local.
import os, sys

if "google.colab" in sys.modules:
    _RAIZ = "/content/S11_series_temporales"
    for _sub in ("notebook", "data", "figuras", "resultados"):
        os.makedirs(os.path.join(_RAIZ, _sub), exist_ok=True)
    os.chdir(os.path.join(_RAIZ, "notebook"))
    print("Colab: carpeta de trabajo en", os.getcwd())


# Sesión 11 — Series temporales — ARIMA/SARIMA, ETS y Prophet

**Curso:** Herramientas para la Ciencia de Datos, Facultad de Negocios, UPC
**Programa:** Administración y Ciencia de Datos para Negocios

> Curso **Herramientas para la Ciencia de Datos**, Facultad de Negocios, Administración y Ciencia de Datos para Negocios — UPC.
> Español impersonal, fechas DD/MM/YYYY, todas las cifras provienen de la investigación verificada de la sesión (`el material de referencia de la sesión`).

Este cuaderno **replica dos resultados seminales del pronóstico** y luego lleva el forecasting a una decisión de negocio, paso a paso:

1. **Modelo aerolínea de Box & Jenkins (1970)** — SARIMA(0,1,1)(0,1,1)[12] sobre `log(AirPassengers)`.
2. **Prophet de Taylor & Letham (2018)** — sobre las vistas de la Wikipedia de Peyton Manning.
3. **Caso de negocio** — forecasting mensual de demanda (Store Item Demand) con **backtesting de origen móvil**.

> **Cómo se abre este cuaderno.** El curso lo distribuye por **Google Drive**: en la
> carpeta compartida, clic derecho sobre el archivo → *Abrir con* → *Google
> Colaboratory*. Conviene empezar por **Archivo → Guardar una copia en Drive** para
> conservar el trabajo. No se requiere cuenta de GitHub ni instalar nada en el equipo:
> los datos de la sesión viajan dentro del propio cuaderno.
> **Carpeta del curso en Drive (Pregrado):** https://drive.google.com/drive/folders/1-YJxRt0n-UZwQCu03Lls2LGUYz6KMsl2


## 1. Objetivos de aprendizaje
Al terminar, el estudiante es capaz de: descomponer una serie en **tendencia, estacionalidad y residuo** (STL); evaluar **estacionariedad** (ADF/KPSS) y decidir la **diferenciación**; leer **ACF/PACF** para proponer un orden ARIMA; ajustar y comparar **ARIMA/SARIMA, ETS/Holt-Winters y Prophet**; validar con **rolling-origin** usando RMSE/MAPE/sMAPE; y **evitar la fuga de información temporal**.

## 2. Mapa de la sesión: nueve capítulos en dos clases

La sesión ocupa **dos clases**. Cada capítulo lleva un código —11.1 a 11.9— que es **el mismo** en el sílabo, en la guía del docente, en la guía de laboratorio y en las diapositivas, de modo que se pueda pasar de un material a otro sin traducir numeraciones.

**JUEVES — 145 min de contenido** (bloque A1 de 75, receso de 15, bloque A2 de 70; antes, 20 min de control sobre la Sesión 10)

| Cód. | Pregunta que responde | Dónde vive en este cuaderno |
|---|---|---|
| **11.1** | ¿Por qué no procede una partición aleatoria cuando los datos tienen fecha? | Sin celdas: se abre en clase con la partición aleatoria que rompe la fecha |
| **11.2** | ¿Qué componentes integran una serie temporal? | «Teoría guiada (minidemostraciones)» — componentes, estacionariedad y selección |
| **11.3** | ¿Qué es la estacionariedad y cómo se alcanza? | «Teoría guiada (minidemostraciones)» — componentes, estacionariedad y selección |
| **11.4** | ¿Cómo se selecciona el modelo de pronóstico? | «Teoría guiada (minidemostraciones)» — componentes, estacionariedad y selección |
| **11.5** | ¿Se sostiene con datos reales? La réplica de Box y Jenkins (1970) | «Modelo aerolínea de Box y Jenkins (1970)» y «Prophet» — laboratorio, pasos 0 a 5 |

**VIERNES — 120 min corridos**

| Cód. | Pregunta que responde | Dónde vive en este cuaderno |
|---|---|---|
| **11.6** | ¿Cuándo se puede confiar en este pronóstico? | «Supuestos: decisiones y condiciones de validez» |
| **11.7** | ¿Cómo se verifica que el resultado es real? | «Verificación desde la base» |
| **11.8** | ¿Qué decisión habilita? Demanda mensual de un producto | «Store Item Demand (forecasting mensual)» — laboratorio, paso 6 |
| **11.9** | ¿Qué no se puede afirmar, y qué sigue en S12? | «Cierre» |

> El **control** de esta sesión se resuelve en aula, en la franja de 20 minutos del jueves siguiente, y cubre **los nueve capítulos**, de los dos días.


## Cómo leer este cuaderno

Este cuaderno no solo **corre**: **explica y descompone** cada paso. Los marcadores guían la lectura:

- **❓ Qué se quiere averiguar** — abre cada resultado importante: la pregunta que ese resultado contesta, qué decisión depende de ella y **qué significaría cada resultado posible, dicho antes de ver el número**. Conviene detenerse ahí y contestar mentalmente antes de ejecutar: un dato solo informa a quien traía una pregunta.
- **🔎 Qué hace este código** — antes de cada celda: qué va a calcular y por qué.
- **📖 Cómo se lee esta salida** — después de una salida numérica clave: cómo interpretarla en negocio.
- **💡 Intuición** y **⚠️ Alerta / supuesto** — matices y errores a evitar.
- **🖐️ Cálculo manual** — se reconstruye la mecánica (la diferenciación `(1−B)(1−Bˢ)` de la log-serie; el MAPE/sMAPE del holdout) y se verifica contra la librería con `assert`.
- **✅ Verificación desde la base** — se **recomputa** el resultado clave desde los datos (coeficientes θ₁/Θ₁, MAPE, Ljung-Box) y se cruza con el Excel (`assert`).
- **🧮 Matemática en el cuerpo** — la fórmula (ARIMA/SARIMA con el operador de rezago `B`; RMSE/MAPE/sMAPE; suavizado exponencial de ETS) donde se aplica.
- **🧱 Construcción desde cero** — se rearma el flujo de pronóstico (log → diferenciar → SARIMA → holdout → MAPE) y se reproduce el contrato (`assert`).
- **📄 En el paper** — procedencia exacta del resultado replicado (autor, año, publicación).

**Convención Excel.** Los resultados y pruebas se vuelcan a `resultados/S11_resultados.xlsx` y las **figuras de resultados se generan LEYENDO ese Excel**. La **verificación desde la base (6.1)**, la **construcción desde cero (Sección 7)** y los **diagnósticos de supuestos (Sección 8) NO escriben en el Excel**.

**Mapa de celdas ↔ slides:** el cuaderno de la sesión. **Supuestos (fuente canónica):** la guía de supuestos de la sesión.

## Preparación del entorno — Setup (Sección 0 del cuaderno)

Una sola celda de instalación para **Google Colab** (núcleo con versiones fijadas; `pmdarima` es opcional y se instala de forma tolerante). En ejecución local se salta automáticamente.

In [ ]:
# SKIP-LOCAL: solo Colab.
# Colab ya trae el nucleo cientifico (numpy, pandas, scipy, matplotlib, seaborn,
# scikit-learn, statsmodels, openpyxl) COMPILADO ENTRE SI. Reinstalarlo con las
# versiones del venv del curso ROMPE el entorno: scipy y statsmodels dejan de
# importar con "cannot import name '_slice' from 'numpy._core.umath'". Por eso
# aqui solo se instala lo que Colab NO trae.
import sys

if "google.colab" in sys.modules:
    %pip install -q prophet sktime

# Trazabilidad (sin reinstalar): versiones en uso frente a la matriz
# certificada del curso en la matriz de versiones certificada del curso. Si alguna difiere, las cifras
# pueden variar en los ultimos decimales; el metodo y las conclusiones no.
import importlib.metadata as _md

_CERTIFICADAS = {
    "statsmodels": "0.14.6",
}

print(f"{'paquete':18}{'en uso':14}{'certificada':14}estado")
for _p, _cert in _CERTIFICADAS.items():
    try:
        _v = _md.version(_p)
    except Exception:
        _v = "ausente"
    _estado = "=" if _v == _cert else "distinta (se respeta la de Colab)"
    print(f"{_p:18}{_v:14}{_cert:14}{_estado}")


**🔎 Qué hace este código.** Importa las librerías base (NumPy, pandas, matplotlib), silencia los avisos de convergencia (**esperados**, no errores), fija las rutas de la sesión (`data/`, `resultados/`, `figuras/`) de forma portable (local, nbconvert y Colab) y define las **métricas de error del curso** con su definición fija: `rmse`, `mape` y `smape`.

In [ ]:
import os, sys, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")            # silencia avisos de convergencia/deprecación
pd.set_option("display.width", 120)
plt.rcParams.update({"figure.figsize": (11, 4), "axes.grid": True,
                     "grid.alpha": 0.30, "figure.dpi": 110, "font.size": 10})

IN_COLAB = "google.colab" in sys.modules

# --- Resolución de rutas (portable local / Colab) ---
_cwd = Path.cwd()
if (_cwd / "data").exists():
    SESSION = _cwd
elif _cwd.name == "notebook" and (_cwd.parent / "data").exists():
    SESSION = _cwd.parent
else:
    SESSION = _cwd
DATA = SESSION / "data"
RESULT = SESSION / "resultados"; RESULT.mkdir(parents=True, exist_ok=True)
FIG = SESSION / "figuras"; FIG.mkdir(parents=True, exist_ok=True)
XLSX = RESULT / "S11_resultados.xlsx"
print("Entorno Colab:", IN_COLAB)
print("Carpeta de sesión:", SESSION)
print("Excel de resultados:", XLSX)

# --- Métricas de error (definiciones fijas del curso) ---
def rmse(a, f):
    a, f = np.asarray(a, float), np.asarray(f, float)
    return float(np.sqrt(np.mean((a - f) ** 2)))

def mape(a, f):
    "Error porcentual absoluto medio, en %."
    a, f = np.asarray(a, float), np.asarray(f, float)
    return float(np.mean(np.abs((a - f) / a)) * 100.0)

def smape(a, f):
    "sMAPE simétrico (definición M4), en % (rango 0-200)."
    a, f = np.asarray(a, float), np.asarray(f, float)
    denom = np.abs(a) + np.abs(f)
    return float(np.mean(2.0 * np.abs(f - a) / denom) * 100.0)

**🔎 Qué hace este código.** Define los **cargadores de datos**: `cargar_airpassengers()` (Serie G de Box & Jenkins, índice mensual), `cargar_peyton()` (CSV oficial de Prophet, columnas `ds`/`y`) y `cargar_store()` (Store Item Demand, filtra una serie y la **agrega a mensual**). Cada uno usa la copia local de `data/` y, si no está, descarga del origen oficial.

In [ ]:
# --- Cargadores de datos (usan data/ local; si no existe, descargan del origen oficial) ---
def cargar_airpassengers():
    f = DATA / "airpassengers.csv"
    if f.exists():
        vals = pd.read_csv(f)["passengers"].astype(float).values
    else:
        from statsmodels.datasets import get_rdataset
        vals = get_rdataset("AirPassengers", "datasets", cache=True).data["value"].astype(float).values
    idx = pd.date_range("1949-01-01", periods=len(vals), freq="MS")
    return pd.Series(vals, index=idx, name="pasajeros")

def cargar_peyton():
    f = DATA / "peyton_manning.csv"
    if f.exists():
        return pd.read_csv(f)
    url = ("https://raw.githubusercontent.com/facebook/prophet/main/"
           "examples/example_wp_log_peyton_manning.csv")
    return pd.read_csv(url)

def cargar_store(store=1, item=1):
    df = None
    for name in ("train.csv", "train_muestra.csv"):
        f = DATA / name
        if f.exists():
            df = pd.read_csv(f, parse_dates=["date"]); break
    if df is None:
        url = ("https://raw.githubusercontent.com/jgonzalezab/"
               "Store-Item-Demand-Forecasting/master/Data/train.csv")
        df = pd.read_csv(url, parse_dates=["date"])
    one = df[(df["store"] == store) & (df["item"] == item)].set_index("date").sort_index()
    return one["sales"].resample("MS").sum()    # agregación mensual (inicio de mes)

## 11.2 a 11.4 — ¿Qué componentes integran una serie temporal, qué es la estacionariedad y cómo se selecciona el modelo de pronóstico? Teoría guiada (minidemostraciones) (Sección 1 del cuaderno)

Una **serie temporal** es una secuencia de observaciones ordenadas y equiespaciadas en el tiempo. Se descompone en **tendencia** (nivel de largo plazo), **estacionalidad** (patrón que se repite con periodo fijo, p. ej. 12 meses), **ciclo** y **residuo**. El modelo puede ser **aditivo** (`y = T + S + R`, la amplitud estacional es constante) o **multiplicativo** (`y = T, S, R`, la amplitud crece con el nivel).

> A diferencia de una muestra de filas intercambiables (S03–S10), aquí las observaciones están **correlacionadas en el tiempo**: esa dependencia es justo la señal que se modela (la guía de supuestos de la sesión, aviso conceptual inicial).

#### 🧮 Matemática en el cuerpo — ARIMA(p,d,q)(P,D,Q)ₛ con el operador de rezago B

Sea $B$ el **operador de rezago**: $B\,y_t = y_{t-1}$ y $B^s y_t = y_{t-s}$. La **diferenciación** que vuelve estacionaria una serie con tendencia y estacionalidad se escribe como productos de $(1-B)$:

$$\nabla y_t=(1-B)\,y_t=y_t-y_{t-1}\ \text{(regular, orden $d$)},\qquad \nabla_s y_t=(1-B^{s})\,y_t=y_t-y_{t-s}\ \text{(estacional, orden $D$)}.$$

Un modelo **SARIMA** $(p,d,q)(P,D,Q)_s$ combina memoria autorregresiva, diferenciación y errores móviles, en sus partes regular y estacional:

$$\underbrace{\phi_p(B)\,\Phi_P(B^{s})}_{\text{AR reg.}\ \times\ \text{AR estac.}}\ \underbrace{(1-B)^{d}\,(1-B^{s})^{D}}_{\text{diferenciación}}\ y_t\;=\;\underbrace{\theta_q(B)\,\Theta_Q(B^{s})}_{\text{MA reg.}\ \times\ \text{MA estac.}}\ \varepsilon_t,\qquad \varepsilon_t\sim\text{ruido blanco}.$$

- El **modelo aerolínea** es $(0,1,1)(0,1,1)_{12}$: no hay parte AR ($p=P=0$); una diferencia regular y una estacional $(1-B)(1-B^{12})$; un MA regular $\theta_1$ y un MA estacional $\Theta_1$. Sobre `log(y)` queda

$$(1-B)(1-B^{12})\,\log y_t=(1+\theta_1 B)(1+\Theta_1 B^{12})\,\varepsilon_t.$$

- **Convención de signos.** `statsmodels` y R `forecast` escriben la parte MA como $(1+\theta B)$ → coeficientes **negativos**; Box & Jenkins (1970) usan $(1-\theta B)$ → los reportan **positivos**: es la **misma** solución. **Invertibilidad:** $|\theta_1|<1$, $|\Theta_1|<1$.

Fuente: el glosario de la sesión puntos 6–9; la guía de supuestos de la sesión puntos 2.3–2.4; la ficha de la sesión de réplica del paper punto 7.

**🔎 Qué hace este código.** Genera dos series sintéticas —una con estacionalidad **aditiva** (amplitud constante) y otra **multiplicativa** (amplitud que crece con el nivel)— y las grafica lado a lado. Es el patrón visual que decide si conviene **transformar con log** antes de modelar.

In [ ]:
# Mini-demo: estacionalidad ADITIVA vs. MULTIPLICATIVA (datos sintéticos, EDA directo)
t = np.arange(1, 121)
tend = 10 + 0.3 * t
estac = np.sin(2 * np.pi * t / 12)
serie_add = tend + 3 * estac                     # amplitud estacional constante
serie_mul = tend * (1 + 0.25 * estac)            # amplitud crece con el nivel

fig, ax = plt.subplots(1, 2, figsize=(11, 3.4), sharex=True)
ax[0].plot(t, serie_add, color="#c8102e"); ax[0].set_title("Estacionalidad ADITIVA")
ax[1].plot(t, serie_mul, color="#c8102e"); ax[1].set_title("Estacionalidad MULTIPLICATIVA")
for a in ax:
    a.set_xlabel("mes"); a.set_ylabel("nivel")
plt.tight_layout(); plt.show()
print("Lectura de negocio: si el pico de diciembre CRECE con las ventas → multiplicativa → conviene log.")

**📖 Cómo se lee.** En la serie **aditiva** el pico estacional suma siempre lo mismo; en la **multiplicativa** el pico **se agranda** con el nivel (efecto embudo). 💡 Regla práctica: si en una serie de ventas el pico de diciembre **crece** año a año en unidades, es multiplicativa → conviene **`log`** para estabilizar la varianza (la guía de supuestos de la sesión, punto 2.2).

### Matriz de decisión ADF / KPSS (sus hipótesis nulas son OPUESTAS)

| ADF (H0: raíz unitaria = NO estacionaria) | KPSS (H0: estacionaria) | Conclusión |
|---|---|---|
| p < 0,05 → rechaza (estacionaria) | p > 0,05 → no rechaza (estacionaria) | **Estacionaria** |
| p > 0,05 → no estacionaria | p < 0,05 → rechaza (no estacionaria) | **No estacionaria → diferenciar** |

⚠️ **Error de lectura frecuente.** Como las nulas se **invierten**, un p-valor pequeño significa conclusiones opuestas en cada prueba: en **ADF** favorece la estacionariedad, en **KPSS** la descarta. Se usan **juntas** para confirmarse (la guía de interpretación de resultados punto 2).

#### 🧮 Matemática en el cuerpo — RMSE, MAPE y sMAPE

Con valores reales $A_t$ y pronósticos $F_t$ sobre $n$ meses del holdout:

$$\text{RMSE}=\sqrt{\tfrac1n\textstyle\sum_t (A_t-F_t)^2},\qquad
\text{MAPE}=\tfrac{100}{n}\textstyle\sum_t\left|\tfrac{A_t-F_t}{A_t}\right|,\qquad
\text{sMAPE}=\tfrac{100}{n}\textstyle\sum_t\frac{2\,|A_t-F_t|}{|A_t|+|F_t|}.$$

- **RMSE** está en las **unidades de la serie** (pasajeros) y penaliza más los errores grandes.
- **MAPE** es adimensional (%), pero **diverge si $A_t\approx 0$** y penaliza más los sobre-pronósticos.
- **sMAPE** es simétrico, acota el problema del denominador y va de **0 a 200 %**; es la **métrica oficial** de la competencia de demanda. ⚠️ No es intercambiable con el MAPE.

Fuente: el glosario de la sesión punto 15; la guía de supuestos de la sesión punto 4.3.

## 11.5 — ¿Se sostiene con datos reales? Replicación — Modelo aerolínea de Box & Jenkins (1970) (Sección 2 del cuaderno)

**Paso 0 — Contexto del paper.** Box, G. E. P. & Jenkins, G. M. (1970). *Time Series Analysis: Forecasting and Control*. Su ejemplo canónico es la **«Serie G»** (AirPassengers): 144 pasajeros mensuales de aerolínea internacional, 1949–1960. Sobre ella ajustaron el **modelo aerolínea** SARIMA(0,1,1)(0,1,1)[12] en logaritmos: dos parámetros que capturan tendencia + estacionalidad multiplicativa. Es una línea base difícil de superar en series mensuales.

> **Protocolo FIJADO al pie (declarado para la sesión de réplica del paper punto 4).** Serie en `np.log`; `SARIMAX(order=(0,1,1), seasonal_order=(0,1,1,12), trend='n', enforce_stationarity=False, enforce_invertibility=False)`; holdout = **últimos 12 meses**; métrica sobre `exp(forecast)`. Cualquier desviación del protocolo cambia los decimales.

### 📄 En el paper

**Procedencia de la réplica** (declarado para la sesión de réplica del paper puntos 1 y 3):

- **Modelo aerolínea SARIMA** — **Box, G. E. P. & Jenkins, G. M. (1970).** *Time Series Analysis: Forecasting and Control.* Holden-Day. Sistematizaron el ciclo **identificación → estimación → diagnóstico**; la **Serie G** (pasajeros mensuales de aerolínea internacional, 1949–1960, 144 obs) es el caso canónico del **SARIMA(0,1,1)(0,1,1)₁₂ sobre `log(y)`**.
- **Lo que se reproduce.** Los coeficientes **`ma.L1` (θ₁)** y **`ma.S.L12` (Θ₁)** negativos y significativos, y un **pronóstico holdout** de 1960 con MAPE bajo.
- **Operativo (venv) vs. benchmark ETIQUETADO.** Valores OPERATIVOS de este cuaderno: **θ₁ ≈ −0,4326** y **Θ₁ ≈ −0,5475** (statsmodels `SARIMAX`, MLE). Benchmark **publicado, otro pipeline**: **θ₁ ≈ −0,4018 / Θ₁ ≈ −0,5569** (R `forecast`/`stats::arima`) — misma solución con distinto optimizador; se cita **etiquetado**, nunca como resultado propio.
- **Datasets.** AirPassengers vía Rdatasets (144 obs). Caso de negocio: **Store Item Demand** (Kaggle), 10 tiendas × 50 productos, agregado a mensual.

### Paso 1 — Cargar AirPassengers e índice mensual

**🔎 Qué hace este código.** Carga las 144 observaciones mensuales, reconstruye el índice temporal (`freq='MS'`) desde ene-1949 y verifica los valores de control de la Serie G (primer 112, último 432, mín 104, máx 622).

In [ ]:
s = cargar_airpassengers()
print(f"AirPassengers: {len(s)} observaciones mensuales | "
      f"{s.index.min():%Y-%m} .. {s.index.max():%Y-%m}")
print(f"Primer valor = {s.iloc[0]:.0f} | último = {s.iloc[-1]:.0f} | "
      f"mín = {s.min():.0f} | máx = {s.max():.0f}")
s.head()

**📖 Cómo se lee.** La serie arranca en **112** (ene-1949) y termina en **432** (dic-1960): 144 meses sin huecos. Estos valores de control confirman que se cargó la **Serie G** original de Box & Jenkins.

**🔎 Qué hace este código.** Grafica la serie cruda para inspeccionar su forma: tendencia y estacionalidad.

In [ ]:
# Gráfico de la serie cruda (EDA directo)
fig, ax = plt.subplots(figsize=(11, 3.6))
ax.plot(s.index, s.values, color="#c8102e")
ax.set_title("AirPassengers — pasajeros mensuales (1949–1960)")
ax.set_xlabel("año"); ax.set_ylabel("pasajeros (miles)")
plt.tight_layout(); plt.show()
print("La amplitud estacional CRECE con el nivel → serie multiplicativa → se modela sobre log.")

**📖 Cómo se lee.** Se ven dos rasgos: una **tendencia creciente** y una **estacionalidad anual** cuya **amplitud crece con el nivel** (los picos recientes oscilan más). ⚠️ Esa amplitud creciente es **no estacionariedad + varianza no constante**: se modela sobre `log(y)` (la guía de supuestos de la sesión, punto 2.2).

### Paso 2 — Descomposición STL y justificación del logaritmo

**🔎 Qué hace este código.** Aplica **STL** (descomposición estacional-tendencia por loess) para separar `trend`/`seasonal`/`resid`, y guarda esos puntos en `df_descomp` para exportarlos al Excel (paso 12). Confirma visualmente por qué conviene el `log`.

In [ ]:
from statsmodels.tsa.seasonal import STL

stl = STL(s, period=12, robust=True).fit()
fig = stl.plot(); fig.set_size_inches(11, 7); plt.tight_layout(); plt.show()

logy = np.log(s)     # transformación log: estabiliza la varianza (serie multiplicativa)
print("Componentes STL calculados (trend / seasonal / resid).")
print("La amplitud del panel 'seasonal' crece con el tiempo → se confirma el log antes de modelar.")

# Se guardan los puntos de la descomposición para exportarlos a Excel (paso 12)
df_descomp = pd.DataFrame({"fecha": s.index, "observado": s.values,
                           "trend": stl.trend.values,
                           "seasonal": stl.seasonal.values,
                           "resid": stl.resid.values})

**📖 Cómo se lee.** El panel `trend` sube de forma sostenida y el `seasonal` muestra un patrón anual cuya **amplitud aumenta** con el tiempo: la estacionalidad es **multiplicativa**. Por eso el modelo aerolínea se ajusta sobre **`log(y)`** (convierte la estructura multiplicativa en aditiva y estabiliza la varianza). 💡 STL es **descriptivo**, no un modelo predictivo; alimenta la fase de modelado.

### Paso 3 — Estacionariedad (ADF y KPSS)

**❓ Qué se quiere averiguar.** ¿Cuánta estructura hay que quitarle a la serie antes de que un modelo ARIMA pueda trabajar con ella, y cómo se sabe que ya basta?

- **Qué decide:** cuántas diferencias entran en el orden del modelo (`d` regular y `D` estacional). Diferenciar de menos deja tendencia sin capturar y sesga el pronóstico; diferenciar de más añade ruido y **ensancha los intervalos** que después se le prometen a quien planifica.
- **Antes de mirar el resultado:** ADF y KPSS tienen hipótesis nulas **opuestas**, así que se leen en pareja. Si sobre la log-serie **en nivel** el ADF no rechaza y el KPSS sí, la serie no es estacionaria y hay que diferenciar. Si **tras** `(1−B)(1−B¹²)` el ADF cae por debajo de 0,05 y el KPSS deja de rechazar, `d=1` y `D=1` bastan y no hace falta una tercera diferencia. Si ambas pruebas siguieran en desacuerdo, la evidencia sería ambigua y lo primero a revisar sería la transformación, no el orden.

**🔎 Qué hace este código.** Corre **ADF** y **KPSS** sobre la log-serie en nivel y sobre la log-serie **diferenciada** (regular `d=1` + estacional `D=1`). Guarda `adf_p_d1D` (target A3).

In [ ]:
from statsmodels.tsa.stattools import adfuller, kpss

def adf_kpss(x, nombre):
    adf_p = float(adfuller(x, autolag="AIC")[1])
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        kpss_p = float(kpss(x, regression="c", nlags="auto")[1])
    print(f"{nombre:38s} ADF p={adf_p:6.4f}  |  KPSS p={kpss_p:6.4f}")
    return adf_p, kpss_p

# Diferenciación regular (d=1) + estacional (D=1) sobre la log-serie
d1  = logy.diff().dropna()
d1D = d1.diff(12).dropna()

print("Serie                                  Pruebas de estacionariedad")
adf_kpss(logy,  "log(AirPassengers)")
adf_p_d1D, kpss_p_d1D = adf_kpss(d1D, "log dif. regular + estacional")
print()
print("Lectura: la log-serie es NO estacionaria (ADF no rechaza, KPSS rechaza).")
print(f"Tras diferenciar (d=1, D=1) la serie ES estacionaria (ADF p={adf_p_d1D:.4f} < 0,05). [A3]")

**📖 Cómo se lee.** En **nivel** la log-serie es **no estacionaria** (ADF no rechaza, KPSS rechaza). Tras diferenciar `(1−B)(1−B¹²)` la serie **es estacionaria**: **ADF p = 0,0002 < 0,05** (target A3). Es el mínimo de diferenciación que estaciona: `d=1`, `D=1` (la guía de supuestos de la sesión, puntos 2.1 y 2.3).

#### 🖐️ Cálculo manual — la diferenciación `(1−B)(1−B¹²)` de la log-serie

**🔎 Qué hace este código.** Reconstruye **de forma manual** con NumPy la diferencia regular `(1−B)` y la estacional `(1−B¹²)` sobre `log(y)`, **verifica con `assert`** que coincide con `pandas .diff().diff(12)`, y corre **ADF** sobre la serie diferenciada manualmente para comprobar que la vuelve **estacionaria** (mismo p-valor del Paso 3).

In [ ]:
# 🖐️ HAZLO A MANO: diferenciar (1-B)(1-B^12) la log-serie y probar estacionariedad (verificación con assert)
from statsmodels.tsa.stattools import adfuller as _adf

lv = np.log(s.values)                    # log de la serie (array)
dreg = lv[1:] - lv[:-1]                   # diferencia REGULAR (1-B): y_t - y_{t-1}
ddob = dreg[12:] - dreg[:-12]             # diferencia ESTACIONAL (1-B^12) sobre la ya diferenciada
ref  = np.log(s).diff().dropna().diff(12).dropna().values   # lo que hace pandas en el Paso 3

assert ddob.shape == ref.shape, (ddob.shape, ref.shape)
assert np.allclose(ddob, ref, atol=1e-10), "la doble diferencia a mano debe igualar a pandas .diff().diff(12)"

adf_p_mano = float(_adf(ddob, autolag="AIC")[1])
print(f"(1-B)(1-B^12) log(y): {len(ddob)} valores | media {ddob.mean():+.5f} (aprox. 0) | var {ddob.var():.5f}")
print(f"ADF sobre la serie diferenciada a mano: p = {adf_p_mano:.4f}   (< 0,05 -> estacionaria)")
print(f"Coincide con statsmodels del Paso 3 (A3):  p = {adf_p_d1D:.4f}")
assert adf_p_mano < 0.05, "la log-serie doblemente diferenciada debe ser estacionaria (ADF)"
assert abs(adf_p_mano - adf_p_d1D) < 1e-6, "la ADF a mano debe coincidir con la del Paso 3"
print("\nassert OK: diferenciar (1-B)(1-B^12) a mano estaciona la serie e iguala a la librería.")

**📖 Cómo se lee.** La doble diferencia manual **iguala** a la de pandas y tiene **media ≈ 0**: la operación `(1−B)(1−B¹²)` quitó tendencia y estacionalidad. Su **ADF p** coincide con el del Paso 3 (**0,0002**): la serie quedó **estacionaria**, condición de entrada de la parte ARMA (la guía de supuestos de la sesión, punto 2.1).

### Paso 4 — ACF/PACF de la doble diferencia → identificar el orden (Drill 1)

**🔎 Qué hace este código.** Traza la **ACF** y la **PACF** de la serie doblemente diferenciada para leer, con las reglas de Box-Jenkins, qué términos MA justifican el orden.

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
plot_acf(d1D, lags=36, ax=ax[0]); ax[0].set_title("ACF — log dif. reg.+estacional")
plot_pacf(d1D, lags=36, ax=ax[1], method="ywm"); ax[1].set_title("PACF — log dif. reg.+estacional")
plt.tight_layout(); plt.show()
print("Pico significativo en el lag 1  → término MA regular q=1.")
print("Pico significativo en el lag 12 → término MA estacional Q=1.")
print("Con d=1 y D=1, esto justifica el orden (0,1,1)(0,1,1)[12] del modelo aerolínea.")

**📖 Cómo se lee.** Un **pico en el lag 1** de la ACF sugiere un **MA regular** (`q=1`); un **pico en el lag 12** sugiere un **MA estacional** (`Q=1`). Con `d=1` y `D=1`, esto identifica el orden **(0,1,1)(0,1,1)[12]** (Drill 1). ⚠️ No sobre-interpretar barras apenas fuera de la banda ±1,96/√n (la guía de supuestos de la sesión, punto 2.4).

### Paso 5 — Ajustar el modelo aerolínea (targets B2, B3, A1, A2)

**❓ Qué se quiere averiguar.** ¿Reproduce este cuaderno el modelo con el que Box y Jenkins pronosticaron el tráfico aéreo en 1970 —**dos parámetros** para 144 meses—, y dicen sus coeficientes lo mismo que la publicación?

- **Qué decide:** si la réplica es válida. Todo lo que viene después —el pronóstico de 1960, su intervalo, la comparación contra ETS y contra Prophet— se apoya en que este ajuste sea el del paper y no otro parecido.
- **Antes de mirar el resultado:** los dos coeficientes MA deben salir **negativos y menores que 1 en valor absoluto**; fuera de ese rango el modelo no sería invertible y no serviría para pronosticar. El benchmark publicado en R es θ₁ ≈ −0,40 y Θ₁ ≈ −0,56: si las cifras del venv caen en esa misma magnitud, la réplica se sostiene y las diferencias son de implementación. Si se apartaran de forma clara, el problema estaría en la transformación o en la diferenciación de esta sesión, no en el paper.

**🔎 Qué hace este código.** Ajusta el **modelo aerolínea** `SARIMAX(log(y), (0,1,1),(0,1,1,12), trend='n')` por máxima verosimilitud e imprime el `summary()` completo.

In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.stats.diagnostic import acorr_ljungbox

mod = SARIMAX(logy, order=(0, 1, 1), seasonal_order=(0, 1, 1, 12), trend="n",
              enforce_stationarity=False, enforce_invertibility=False)
res = mod.fit(disp=False)
print(res.summary())

**🔎 Qué hace este código.** Extrae del ajuste los cuatro números clave: los coeficientes **θ₁** (`ma.L1`, B2) y **Θ₁** (`ma.S.L12`, B3), el **AIC** (A1) y el **p-valor de Ljung-Box(12)** de los residuos (A2).

In [ ]:
ma1  = float(res.params["ma.L1"])        # target B2 (θ₁)
sma1 = float(res.params["ma.S.L12"])     # target B3 (Θ₁)
aic  = float(res.aic)                     # A1
lb_p = float(acorr_ljungbox(res.resid.iloc[13:], lags=[12],
                            return_df=True)["lb_pvalue"].iloc[-1])   # A2

print(f"[B2] ma.L1   (θ₁) = {ma1:+.4f}   —  benchmark publicado R: −0,4018 (otro pipeline)")
print(f"[B3] ma.S.L12(Θ₁) = {sma1:+.4f}   —  benchmark publicado R: −0,5569 (otro pipeline)")
print(f"[A1] AIC (log-serie)        = {aic:.2f}")
print(f"[A2] Ljung-Box(12) p-valor  = {lb_p:.3f}  (>0,05 → residuos ≈ ruido blanco: buen ajuste)")
print()
print("Signos negativos por la convención (1+θL) de statsmodels/R; Box & Jenkins (1−θL) los")
print("reporta positivos: es la MISMA solución. Lo estable es |θ₁|≈0,43 y |Θ₁|≈0,55.")

**📖 Cómo se lee.** Los coeficientes operativos son **θ₁ = −0,4326** y **Θ₁ = −0,5475**, negativos e invertibles, como en la réplica: el **benchmark publicado de R** (−0,4018 / −0,5569) cae en la misma magnitud y se cita **etiquetado**. El **AIC = −435,44** solo compara modelos sobre la **misma** log-serie. El **Ljung-Box(12) p = 0,754 > 0,05** dice que los **residuos son ruido blanco**: el ajuste capturó la estructura (la guía de supuestos de la sesión, puntos 2.4 y 2.5). 💡 Los signos negativos son por la convención `(1+θB)`; Box & Jenkins los reporta positivos: **misma** solución.

#### 🧮 Matemática en el cuerpo — suavizado exponencial y ETS (Holt-Winters)

El **suavizado exponencial** pronostica con un **promedio ponderado que da más peso a lo reciente**. En su forma simple, con parámetro de nivel $\alpha\in(0,1)$:

$$\ell_t=\alpha\,y_t+(1-\alpha)\,\ell_{t-1}\quad\Longleftrightarrow\quad \hat y_{t+1}=\alpha\sum_{j\ge 0}(1-\alpha)^{j}\,y_{t-j}.$$

**Holt-Winters** añade **tendencia** $b_t$ (parámetro $\beta$) y **estacionalidad** $s_t$ de periodo $m$ (parámetro $\gamma$). En su forma **aditiva**:

$$\ell_t=\alpha\,(y_t-s_{t-m})+(1-\alpha)(\ell_{t-1}+b_{t-1}),\quad b_t=\beta\,(\ell_t-\ell_{t-1})+(1-\beta)b_{t-1},\quad s_t=\gamma\,(y_t-\ell_t)+(1-\gamma)s_{t-m},$$
$$\hat y_{t+h}=\ell_t+h\,b_t+s_{t-m+h}.$$

La versión **multiplicativa** reemplaza las sumas del componente estacional por productos ($y_t/s_{t-m}$, $s_{t-m+h}$ multiplicando). A diferencia de ARIMA, **ETS no exige estacionariedad** (no diferencia): modela los componentes directamente. Fuente: el glosario de la sesión punto 11; la guía de supuestos de la sesión punto 3.1.

### Paso 6 — Contraste con `auto_arima` y ETS/Holt-Winters (Drill 2)

**❓ Qué se quiere averiguar.** Si una búsqueda automática propone un orden **distinto** del que se identificó manualmente con la ACF y la PACF, ¿hay que cambiar de modelo?

- **Qué decide:** cuál de los dos modelos se lleva al pronóstico y, sobre todo, **quién podrá corregirlo cuando falle**. Un orden identificado manualmente se explica, se audita y se reajusta ante un cambio de patrón; un orden que salió de una búsqueda por AIC solo se puede volver a lanzar y aceptar lo que devuelva.
- **Antes de mirar el resultado:** el modelo aerolínea tiene **AIC = −435,44**. Si la búsqueda devuelve un orden con el **mismo bloque estacional** `(0,1,1)[12]` y un AIC apenas mejor —un par de puntos sobre una escala de cientos—, coincide con la identificación manual y esa diferencia no compra nada que justifique perder la trazabilidad. Si devolviera un orden **radicalmente distinto** con una mejora grande, la lectura de la ACF/PACF habría fallado y habría que rehacerla, no taparla con la automática.

**🔎 Qué hace este código.** Contrasta el orden fijado con una **búsqueda automática**: usa `pmdarima.auto_arima` si está disponible y, si no, hace una **mini-búsqueda por AIC** con `SARIMAX` sobre órdenes vecinos del aerolínea. Es una **confirmación** de la identificación manual, no un sustituto.

In [ ]:
# Contraste con auto_arima (pmdarima). Si el paquete no está disponible en el entorno,
# se compara por AIC unos pocos órdenes VECINOS del aerolínea con SARIMAX (misma log-serie).
try:
    import pmdarima as pm
    auto = pm.auto_arima(logy, seasonal=True, m=12, stepwise=True,
                         suppress_warnings=True, error_action="ignore", trace=False)
    orden_auto, sorden_auto, aic_auto = auto.order, auto.seasonal_order, float(auto.aic())
    fuente = "pmdarima.auto_arima"
except Exception as e:
    candidatos = [((0, 1, 1), (0, 1, 1, 12)), ((1, 1, 1), (0, 1, 1, 12)),
                  ((1, 1, 0), (0, 1, 1, 12)), ((0, 1, 2), (0, 1, 1, 12)),
                  ((0, 1, 1), (1, 1, 0, 12))]
    tabla = []
    for od, so in candidatos:
        r = SARIMAX(logy, order=od, seasonal_order=so, trend="n",
                    enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
        tabla.append((od, so, float(r.aic)))
    orden_auto, sorden_auto, aic_auto = min(tabla, key=lambda t: t[2])
    fuente = f"búsqueda por AIC con SARIMAX (pmdarima no disponible: {type(e).__name__})"

print(f"Mejor orden [{fuente}]: {orden_auto} estacional {sorden_auto}  (AIC {aic_auto:.2f})")
print("Confirma un modelo cercano al aerolínea (0,1,1)(0,1,1)[12]; contraste, no sustituye al orden fijado.")

**📖 Cómo se lee.** La búsqueda automática recupera un orden **cercano al aerolínea** (mismo bloque estacional `(0,1,1)[12]`), lo que **confirma** la identificación manual de ACF/PACF. ⚠️ El AIC solo compara modelos sobre la **misma** transformación y diferenciación; no se compara entre órdenes con distinto `d`.

**🔎 Qué hace este código.** Ajusta **ETS Holt-Winters** en sus dos formas —**aditiva** y **multiplicativa**— sobre la serie en nivel y compara su AIC (Drill 2: elegir el tipo estacional).

In [ ]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing

ets_add = ExponentialSmoothing(s, trend="add", seasonal="add", seasonal_periods=12).fit()
ets_mul = ExponentialSmoothing(s, trend="add", seasonal="mul", seasonal_periods=12).fit()
print(f"ETS Holt-Winters ADITIVO        AIC = {ets_add.aic:.2f}")
print(f"ETS Holt-Winters MULTIPLICATIVO AIC = {ets_mul.aic:.2f}")
print()
print("Drill 2: en AirPassengers la amplitud estacional crece con el nivel, así que el")
print("MULTIPLICATIVO ajusta mejor (menor AIC). Aditivo convendría si el pico sumara siempre lo mismo.")

**📖 Cómo se lee.** En AirPassengers la amplitud estacional **crece con el nivel**, así que el **ETS multiplicativo** ajusta mejor (menor AIC). El **aditivo** convendría si el pico sumara siempre lo mismo (la guía de supuestos de la sesión, puntos 2.2 y 3.1). 💡 ETS **no exige estacionariedad**: es una línea base rápida y a menudo competitiva con SARIMA.

### Paso 7 — Pronóstico holdout (12 meses) → target B4 (MAPE) y RMSE

**❓ Qué se quiere averiguar.** Con la información disponible a diciembre de 1959, ¿con qué error se habría pronosticado **todo 1960**? Y, más importante para quien planifica, ¿qué **margen** habría que reservar mes a mes para que la capacidad no resulte insuficiente?

- **Qué decide:** la banda, no el punto. Quien dimensiona flota, inventario o turnos no compromete capacidad con la media pronosticada: usa el **límite superior** del intervalo cuando el faltante es el costo dominante, o el inferior cuando lo caro es el exceso de stock.
- **Antes de mirar el resultado:** se pronostican 12 meses hacia adelante, siempre hacia el futuro y nunca con meses sorteados al azar. Si el MAPE superara el 10 %, el pronóstico no serviría para comprometer capacidad; si queda del orden del 3 %, el modelo clásico es utilizable tal cual. Y conviene mirar el **ancho** del intervalo del 95 %: si a un mes de horizonte es estrecho y a doce meses se abre a más de cien pasajeros, la banda aporta una advertencia accionable —la incertidumbre crece con el horizonte—, y comprometerse a un año con la holgura de un mes es lo que invalida la planificación.

**🔎 Qué hace este código.** Reentrena el modelo aerolínea con los **primeros 132 meses**, pronostica los **últimos 12** (1960), des-transforma con `exp` y calcula **RMSE** y **MAPE** en la escala de pasajeros (con su intervalo del 95 %). Guarda `df_holdout`.

In [ ]:
train, test = s.iloc[:-12], s.iloc[-12:]     # holdout = último año (1960)
res_h = SARIMAX(np.log(train), order=(0, 1, 1), seasonal_order=(0, 1, 1, 12), trend="n",
                enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)

fc   = res_h.get_forecast(steps=12)
pred = np.exp(fc.predicted_mean)                 # des-transformar con exp → escala de pasajeros
ci   = np.exp(fc.conf_int(alpha=0.05))
pred.index = test.index; ci.index = test.index

rmse_holdout = rmse(test.values, pred.values)
mape_holdout = mape(test.values, pred.values)    # target B4 (en %)
print(f"[B4] MAPE holdout 12 meses = {mape_holdout:.2f} %  ,   RMSE = {rmse_holdout:.2f} pasajeros")
print("El modelo aerolínea pronostica 1960 con error bajo: la línea base clásica sigue siendo fuerte.")

df_holdout = pd.DataFrame({"fecha": test.index, "real": test.values, "pred": pred.values,
                           "lower": ci.iloc[:, 0].values, "upper": ci.iloc[:, 1].values})
df_holdout

**📖 Cómo se lee.** El **MAPE del holdout = 2,88 %** (target B4) y el **RMSE ≈ 18,5 pasajeros**: el modelo aerolínea de **dos parámetros** pronostica 1960 con error bajo. La línea base clásica sigue siendo fuerte. El **intervalo** se ensancha con el horizonte: es la información que dimensiona el stock (la guía de supuestos de la sesión, punto 4.3).

#### 🖐️ Cálculo manual — MAPE y sMAPE del holdout desde el pronóstico

**🔎 Qué hace este código.** Recompone **manualmente** el **MAPE** y el **sMAPE** del holdout desde los 12 valores reales y pronosticados, y **verifica con `assert`** que coinciden con los ayudantes `mape()`/`smape()` y con el target B4 (≈ 2,88 %).

In [ ]:
# 🖐️ HAZLO A MANO: MAPE y sMAPE del holdout desde el pronóstico vs. lo real (verificación con assert)
real_h = test.values.astype(float)      # 12 valores reales de 1960
pred_h = pred.values.astype(float)      # pronóstico del modelo aerolínea (des-transformado con exp)

ape = np.abs(real_h - pred_h) / real_h                          # error porcentual absoluto por mes
mape_mano = float(ape.mean() * 100.0)
sape = 2.0 * np.abs(real_h - pred_h) / (np.abs(real_h) + np.abs(pred_h))
smape_mano = float(sape.mean() * 100.0)

print(f"MAPE a mano  = {mape_mano:.4f} %   (helper mape()  = {mape(real_h, pred_h):.4f} %)")
print(f"sMAPE a mano = {smape_mano:.4f} %   (helper smape() = {smape(real_h, pred_h):.4f} %)")
print(f"Excel B4 (MAPE holdout, Paso 7) = {mape_holdout:.4f} %")
assert abs(mape_mano - mape(real_h, pred_h)) < 1e-9, "el MAPE a mano debe igualar al helper"
assert abs(mape_mano - mape_holdout) < 1e-9, "el MAPE a mano debe igualar al del Paso 7"
print("\nassert OK: el MAPE/sMAPE calculado a mano reproduce el del cuaderno (aprox. 2,88 %).")

**📖 Cómo se lee.** El **MAPE calculado manualmente (2,8836 %)** coincide con el helper y con el Paso 7: es el **promedio de los 12 errores porcentuales** mes a mes. El **sMAPE** (simétrico) da un valor cercano pero **no idéntico** —usa `|A|+|F|` en el denominador—: por eso no se reportan como si fueran la misma métrica (la guía de supuestos de la sesión, punto 4.3).

## 11.5 (continúa) — Replicación — Prophet de Taylor & Letham (2018) (Sección 3 del cuaderno)

**Contexto.** Taylor, S. J. & Letham, B. (2018). *Forecasting at Scale*. The American Statistician 72(1):37-45. Prophet modela la serie como suma de **tendencia por tramos** (con changepoints) + **estacionalidades de Fourier** + **festivos**. Su ejemplo oficial son las **vistas diarias (log) de la Wikipedia de Peyton Manning** (~8 años, con fuerte estacionalidad semanal/anual y picos por partidos).

### Qué preguntaba Prophet, y por qué usó lo que usó — Sección 0 del paper (subsección 3.0)

**💡 Antes de ajustar nada.** Una réplica sin esta pregunta se vuelve mecánica: se ejecutan celdas y sale un número. Lo que sigue explica **qué buscaban los autores** y **por qué eligieron cada pieza de su método**, que es de donde proviene el criterio para elegir un método propio mañana. *(Desarrollo completo con las citas del preprint: la ficha de la sesión de réplica del paper, «Sección 0».)*

**El problema declarado no era de precisión, era de organización.** El resumen del artículo lo dice sin rodeos: «hay retos serios asociados a la producción de pronósticos fiables y de alta calidad, sobre todo cuando existe una variedad de series temporales y los analistas con experiencia en modelado de series son relativamente escasos» (p. 1). La pregunta era **organizacional**: ¿cómo produce una empresa pronósticos confiables de miles de series cuando quien conoce el negocio no sabe de series temporales y los especialistas en series temporales son muy escasos? De ahí el título del paper — «la demanda de pronósticos de calidad supera con mucho el ritmo al que pueden producirse» (p. 2)—. Y **«escala» no es cómputo ni almacenamiento**: los autores descartan esa lectura de forma explícita (p. 3) y la sustituyen por muchas personas sin formación en series, muchos problemas distintos y la necesidad de confiar en cientos de pronósticos ya producidos.

**Las alternativas existían, y se midieron antes de descartarlas.** En 2017 el paquete `forecast` de R ya ofrecía `auto.arima`, `ets`, `snaive` y `tbats` (p. 5). Los autores no supusieron su fracaso: corrieron los cuatro procedimientos sobre la serie diaria de eventos de Facebook en **tres fechas de corte distintas**, cada una con solo la historia disponible ese día (Fig. 3, pp. 5-6). El ARIMA automático comete errores grandes de tendencia cuando la tendencia cambia cerca del corte y no captura la estacionalidad; el suavizado exponencial y el ingenuo estacional captan el ciclo semanal pero pierden el anual; todos sobrerreaccionan a la caída de fin de año (p. 5).

**El criterio de descarte fue la reparabilidad, no la precisión.** Aquí está el matiz que da sentido a toda la sección: «los primeros parámetros de entrada del ARIMA automático son los órdenes máximos de diferenciación, de los componentes autorregresivos y de los de media móvil. Un analista típico no sabrá cómo ajustar esos órdenes para evitar el comportamiento de la Fig. 3 — este es el tipo de pericia que resulta difícil de escalar» (p. 5). Los mandos de un SARIMA son (p, d, q)(P, D, Q), órdenes de un proceso generativo sin traducción al lenguaje del negocio: quien no los interpreta tampoco los corrige, y a escala **un modelo incorregible es un modelo inservible**. La Sección 2 de este cuaderno acaba de mostrar el trabajo experto que exige esa identificación (ACF/PACF, Ljung-Box); el paper no lo niega, lo declara **no escalable**.

**De ahí la decisión estructural — ajuste de curva en lugar de proceso generativo.** «Planteamos, en efecto, el problema de pronóstico como un ejercicio de ajuste de curva […]. Aunque renunciamos a ventajas inferenciales importantes del uso de un modelo generativo como ARIMA, esta formulación aporta una serie de ventajas prácticas» (p. 7). El modelo es la suma `y(t) = g(t) + s(t) + h(t) + ε_t` (ecuación 1, p. 7), un modelo aditivo generalizado con el **tiempo como único regresor**. Lo renunciado: la dependencia temporal explícita, la identificación por ACF/PACF, el diagnóstico sobre residuos correlacionados. Lo ganado (pp. 7-8): varias estacionalidades a la vez, tolerancia a huecos sin interpolación, ajuste rápido para iterar y «parámetros fácilmente interpretables que el analista puede cambiar para imponer supuestos sobre el pronóstico», porque «los analistas sí suelen tener experiencia con la regresión». Los tres componentes que se verán en `plot_components` son, en el fondo, tres preguntas de negocio: **¿cuándo cambió la tendencia?** (changepoints, con el techo `C(t)` como tamaño de mercado, p. 9), **¿qué ciclos tiene?** (Fourier, periodo 365,25 y 7, pp. 11-12) y **¿qué días son especiales?** (festivos, que van aparte porque «sus efectos no se modelan bien con un ciclo suave», p. 12).

**⚠️ Diferencia con el paper — lo que este cuaderno replica y lo que no.** Aquí **no se replica ningún coeficiente ni ninguna tabla del artículo**, y es deliberado. Primero, porque nadie decide nada con el valor de un coeficiente de Fourier: la pregunta era organizacional y la única cifra que la contesta es el **error fuera de muestra a lo largo del horizonte**, medido con pronósticos históricos simulados — que es exactamente el target de esta sección. Segundo, porque la serie que recorre el artículo es la de **eventos internos de Facebook**, que no es pública; se usa en su lugar el **ejemplo oficial del paquete** (Peyton Manning), que reúne los mismos rasgos de la Fig. 2. En consecuencia, lo que se reproduce es el **procedimiento** del paper, y no hay en el preprint una cifra de MAPE con la que contrastar el resultado. Y una advertencia gemela: el MAPE de esta sección y el del holdout de AirPassengers **no son comparables** — son series distintas, con frecuencias y dificultades distintas.


### 📄 En el paper

**Procedencia de la réplica** (declarado para la sesión de réplica del paper puntos 1.2 y 3):

- **Prophet** — **Taylor, S. J. & Letham, B. (2018).** *Forecasting at Scale.* **The American Statistician 72(1):37-45.** DOI 10.1080/00031305.2017.1380080 (preprint abierto PeerJ 3190). Modelo **aditivo por componentes** pensado para que analistas de negocio produzcan pronósticos a escala.
- **Lo que se reproduce.** El ejemplo oficial (`ds`/`y`) y su **backtesting** con `cross_validation(initial='730 days', period='180 days', horizon='365 days')` → 11 pronósticos; `performance_metrics` → **MAPE por horizonte**.
- **Operativo (venv) vs. benchmark ETIQUETADO.** Valor OPERATIVO: **MAPE medio de `cross_validation` = 7,45 %** (5,84 % a horizonte corto, 10,04 % a un año). Benchmark **doc oficial de Prophet, otro pipeline**: **~5 % a 1 mes, ~11 % a 1 año**.
- **Aviso central.** Más sofisticación **no** garantiza mejor pronóstico: el modelo aerolínea de dos parámetros logra 2,88 % en su holdout; son series distintas, pero ilustran que la parsimonia de SARIMA suele ser difícil de superar en series mensuales (la guía de supuestos de la sesión, punto 3.2).

### Paso 8 — Ajustar Prophet, pronosticar 365 días y ver componentes — Paso 9 — Backtesting con `cross_validation` → target B5

**❓ Qué se quiere averiguar.** Cuando un proveedor presenta «nuestro modelo tiene 7 % de error», ¿a qué **horizonte** corresponde esa cifra? ¿Vale lo mismo para pronosticar la semana que viene que para pronosticar dentro de un año?

- **Qué decide:** si se puede firmar un compromiso con ese número. Un MAPE promedio esconde que el error a corto plazo y el de dentro de doce meses no son la misma magnitud, y el negocio decide siempre a un horizonte concreto.
- **Antes de mirar el resultado:** `cross_validation` reentrena en varios cortes y evalúa **hacia adelante**, y devuelve el error **por horizonte**. Si el MAPE fuera plano a lo largo del horizonte, un único número resumiría bien el desempeño y la discusión sobraría. Si **crece** con el horizonte —del orden de 6 % a corto plazo frente a cerca de 10 % a un año—, entonces reportar solo el promedio sobrestima la precisión de los horizontes largos y subestima la de los cortos: lo que hay que llevar a la reunión es el error **al horizonte de la decisión**.

**🔎 Qué hace este código.** Ajusta `Prophet()` sobre Peyton Manning, pronostica **365 días**, muestra el ajuste y sus **componentes** (tendencia + changepoints + estacionalidades) y corre el **backtesting** oficial con `cross_validation` → **MAPE por horizonte** (target B5). Prophet se envuelve en `try/except`: el target #4 es **opcional** (SARIMA/ETS no dependen de él).

In [ ]:
# Prophet se envuelve en try/except: informa con claridad si el backend Stan fallara.
# El target #4 (Prophet) es OPCIONAL; SARIMA/ETS no dependen de él.
prophet_mape = None
perf = None
prophet_cv = None
try:
    import prophet
    # Cinturón de seguridad (Windows): fija CMDSTAN al bundled si no está definido.
    os.environ.setdefault("CMDSTAN", os.path.join(os.path.dirname(prophet.__file__),
                                                  "stan_model", "cmdstan-2.37.0"))
    from prophet import Prophet
    from prophet.diagnostics import cross_validation, performance_metrics

    dfp = cargar_peyton()
    print(f"Peyton Manning: {dfp.shape[0]} filas | columnas {list(dfp.columns)} | "
          f"{dfp['ds'].min()} .. {dfp['ds'].max()}")

    m = Prophet()
    m.fit(dfp)

    future = m.make_future_dataframe(periods=365)
    fcst = m.predict(future)

    fig1 = m.plot(fcst); plt.title("Prophet — ajuste y pronóstico a 365 días"); plt.show()
    fig2 = m.plot_components(fcst); plt.show()

    # Backtesting: 11 pronósticos sobre ~8 años (protocolo de la documentación oficial)
    df_cv = cross_validation(m, initial="730 days", period="180 days", horizon="365 days")
    perf = performance_metrics(df_cv)
    prophet_mape = float(perf["mape"].mean() * 100.0)          # target B5 (en %)
    prophet_cv = pd.DataFrame({"horizonte_dias": perf["horizon"].dt.days,
                               "mape_pct": perf["mape"] * 100.0})
    print(f"\n[B5] Prophet MAPE medio de cross_validation = {prophet_mape:.2f} %")
    print("     Benchmark doc oficial (otro pipeline): ~5 % a 1 mes, ~11 % a 1 año.")
    print("     El error CRECE con el horizonte: predecir a 1 año es más incierto que a 1 mes.")
except Exception as e:
    print(f"[B5] Prophet no disponible en este entorno: {type(e).__name__}: {e}")
    print("     Target #4 opcional; la réplica SARIMA/ETS y el caso de negocio no se ven afectados.")

**📖 Cómo se lee.** El **MAPE medio de `cross_validation` = 7,45 %** (target B5). Lo revelador es que el error **crece con el horizonte** (≈ 5,8 % a horizonte corto, ≈ 10 % a un año): reportar un único MAPE **sin decir a qué horizonte** es un error de lectura (la guía de supuestos de la sesión, punto 3.2). ⚠️ Los componentes de Prophet son **descriptivos, no causales**.

## 11.8 — ¿Qué decisión habilita? Laboratorio de negocio — Store Item Demand (forecasting mensual) (Sección 4 del cuaderno)

**Store Item Demand Forecasting** (Kaggle): 5 años de ventas diarias, 10 tiendas × 50 productos. Se elige una serie (**tienda 1, producto 1**), se **agrega a mensual** (60 meses) y se comparan tres modelos con **backtesting de origen móvil**. Métrica oficial de la competencia: **sMAPE**.

### Paso 10 — Cargar Store Demand, serie (tienda 1, producto 1), agregación mensual

**🔎 Qué hace este código.** Carga la serie de la **tienda 1, producto 1**, la **agrega a mensual** (`resample('MS').sum()`) y la grafica para inspeccionar su patrón.

In [ ]:
monthly = cargar_store(store=1, item=1)
print(f"Serie mensual (tienda 1, producto 1): {len(monthly)} meses | "
      f"{monthly.index.min():%Y-%m} .. {monthly.index.max():%Y-%m}")

fig, ax = plt.subplots(figsize=(11, 3.4))
ax.plot(monthly.index, monthly.values, marker="o", ms=3, color="#c8102e")
ax.set_title("Store Demand — ventas mensuales (tienda 1, producto 1)")
ax.set_xlabel("año"); ax.set_ylabel("unidades / mes")
plt.tight_layout(); plt.show()
print("Tendencia creciente + estacionalidad anual: apta para SARIMA/ETS y backtesting rolling-origin.")

**📖 Cómo se lee.** La serie mensual (60 meses) muestra **tendencia creciente + estacionalidad anual**: es apta para SARIMA/ETS y para un **backtesting de origen móvil** con periodo estacional 12. 💡 La agregación diaria→mensual reduce ruido y alinea la serie con la decisión de negocio (planificación mensual).

### Paso 11 — Backtesting de origen móvil: seasonal-naive vs. ETS vs. SARIMA (Drill 3)

**Sin fuga temporal (leakage).** El origen se **avanza en el tiempo**: cada modelo se entrena solo con el pasado y se evalúa en el futuro inmediato. Nunca se usa K-fold aleatorio (mezclaría futuro y pasado). El **seasonal-naive** (repetir el mismo mes del año anterior) es la línea base obligatoria: un modelo que no lo supera, no aporta.

**❓ Qué se quiere averiguar.** Para planificar la reposición de un producto, ¿aporta algo un modelo estadístico frente a la regla que la tienda ya aplica —«pedir lo mismo que el año pasado en este mes»?

- **Qué decide:** si el proyecto se justifica. Un modelo que no supera al **seasonal-naive** no merece el costo de mantenerlo ni el riesgo de que nadie sepa repararlo, por sofisticado que sea.
- **Antes de mirar el resultado:** las tres familias se evalúan bajo el mismo esquema de origen móvil —entrenar con el pasado, medir sobre los 6 meses siguientes— y con el **sMAPE**, la métrica oficial de la competencia. Si el seasonal-naive quedara primero o empatado, la recomendación sería no cambiar nada. Si SARIMA o ETS bajan su sMAPE en un punto porcentual o más, la mejora es real y se traduce en unidades de inventario. Y conviene revisar la tabla por origen: un campeón que presenta el menor error en **todos** los orígenes es una recomendación; uno que lo logra solo en uno es una casualidad.

**🔎 Qué hace este código.** Implementa el **rolling-origin** (`initial=36, horizon=6, step=6, m=12`): en cada origen entrena con el pasado, pronostica 6 meses y mide **sMAPE** de **seasonal-naive**, **ETS** y **SARIMA**; promedia sobre orígenes y elige el **campeón** (targets A4 y B6). Guarda `df_backtest`.

**⚙️ De dónde proviene el orden `SARIMA(1,0,0)(0,1,1)₁₂` de la serie Store (honestidad metodológica).** A diferencia del **modelo aerolínea**, cuyo `(0,1,1)(0,1,1)₁₂` se identificó **leyendo la ACF/PACF** de la doble diferencia (Paso 4), aquí el orden **no se identifica visualmente** sobre la serie Store. Se fija una sola vez por **selección automática guiada por AIC** —equivalente a `pmdarima.auto_arima(seasonal=True, m=12)` sobre esta serie mensual, o a la mini-búsqueda por AIC del Paso 6— y se **reutiliza en todos los orígenes** del backtesting. Cablear el orden ya elegido, en vez de reidentificarlo en cada corte, mantiene un **protocolo fijo y reproducible** para comparar las tres familias y **no introduce fuga**: la identificación no mira el futuro de cada origen. La diferencia estacional `D=1` es la que captura el ciclo anual de 12 meses (la guía de supuestos de la sesión, puntos 2.3-2.4). No es una caja negra: es una decisión de protocolo, declarada aquí.

In [ ]:
initial, horizon, step, m_ = 36, 6, 6, 12     # ventana expansiva; horizonte 6 meses
res_sm = {"snaive": [], "ets": [], "sarima": []}
filas = []

for o in range(initial, len(monthly) - horizon + 1, step):
    tr, te = monthly.iloc[:o], monthly.iloc[o:o + horizon]     # split que respeta el tiempo

    # 1) seasonal-naive: mismo mes del año anterior (solo mira el pasado)
    snaive = np.array([tr.iloc[len(tr) - m_ + i] for i in range(horizon)], float)
    # 2) ETS Holt-Winters aditivo
    try:
        ets_fc = ExponentialSmoothing(tr, trend="add", seasonal="add",
                                      seasonal_periods=m_).fit().forecast(horizon).to_numpy()
    except Exception:
        ets_fc = snaive.copy()
    # 3) SARIMA
    try:
        sar_fc = SARIMAX(tr, order=(1, 0, 0), seasonal_order=(0, 1, 1, m_),
                         enforce_stationarity=False,
                         enforce_invertibility=False).fit(disp=False).forecast(horizon).to_numpy()
    except Exception:
        sar_fc = snaive.copy()

    real = te.to_numpy()
    res_sm["snaive"].append(smape(real, snaive))
    res_sm["ets"].append(smape(real, ets_fc))
    res_sm["sarima"].append(smape(real, sar_fc))
    filas.append({"origen": f"{monthly.index[o]:%Y-%m}",
                  "snaive": res_sm["snaive"][-1], "ets": res_sm["ets"][-1],
                  "sarima": res_sm["sarima"][-1]})

means = {k: float(np.mean(v)) for k, v in res_sm.items()}
smape_naive, smape_ets, smape_sarima = means["snaive"], means["ets"], means["sarima"]
champ = min(("ets", "sarima"), key=lambda k: means[k])
smape_champ = means[champ]     # target B6

df_backtest = pd.DataFrame(filas)
print(f"[A4] sMAPE seasonal-naive (línea base) = {smape_naive:.2f} %")
print(f"     sMAPE ETS                          = {smape_ets:.2f} %")
print(f"     sMAPE SARIMA                       = {smape_sarima:.2f} %")
print(f"[B6] Campeón = {champ.upper()}  sMAPE = {smape_champ:.2f} %  "
      f"(bate al naive estacional {smape_naive:.2f} %)")
df_backtest

**📖 Cómo se lee.** El **campeón SARIMA (sMAPE 4,09 %, B6)** supera al **ETS (5,50 %)** y al **seasonal-naive (5,78 %, A4)**: aporta valor sobre la línea base. Como el error se promedia sobre **varios orígenes** que respetan el tiempo, es una estimación **honesta** del desempeño en producción (la guía de supuestos de la sesión, punto 4.2).

## Transversal — Exportación a Excel (convención del curso) (Sección 5 del cuaderno)

Todos los resultados van a `resultados/S11_resultados.xlsx`. La hoja **`series_airpassengers`** concentra los targets B2–B6; hojas adicionales guardan la descomposición, el pronóstico y el backtesting para trazar las gráficas.

**🔎 Qué hace este código.** **Escribe el Excel de contrato**: vuelca `resumen` (B2–B6 + filas de apoyo A1–A4, RMSE) a la hoja `series_airpassengers` y las hojas `descomposicion`, `pronostico_holdout`, `backtesting_store` y `prophet_cv`. Es la **única** celda que escribe el Excel; las secciones 6.1, 7 y 8 solo lo leen.

In [ ]:
resumen = pd.DataFrame({
    "metrica": [
        "ma.L1 (theta1) modelo aerolinea",           # B2
        "ma.S.L12 (Theta1) estacional",              # B3
        "MAPE holdout 12m AirPassengers (%)",        # B4
        "Prophet MAPE CV medio Peyton (%)",          # B5
        "sMAPE campeon Store Demand (%)",            # B6
        "AIC modelo aerolinea (log-serie)",          # A1
        "Ljung-Box(12) p-valor residuos",            # A2
        "ADF p-valor (log dif reg+estacional)",      # A3
        "RMSE holdout 12m (pax)",
        "sMAPE seasonal-naive Store Demand (%)",     # A4
        "sMAPE ETS Store Demand (%)",
    ],
    "valor": [ma1, sma1, mape_holdout, prophet_mape, smape_champ,
              aic, lb_p, adf_p_d1D, rmse_holdout, smape_naive, smape_ets],
})

with pd.ExcelWriter(XLSX, engine="openpyxl") as xw:
    resumen.to_excel(xw, sheet_name="series_airpassengers", index=False)
    df_descomp.to_excel(xw, sheet_name="descomposicion", index=False)
    df_holdout.to_excel(xw, sheet_name="pronostico_holdout", index=False)
    df_backtest.to_excel(xw, sheet_name="backtesting_store", index=False)
    if prophet_cv is not None:
        prophet_cv.to_excel(xw, sheet_name="prophet_cv", index=False)

print("Excel escrito en:", XLSX)
print("Hoja 'series_airpassengers' → B2..B6 =",
      [round(x, 4) if isinstance(x, float) else x
       for x in [ma1, sma1, mape_holdout, prophet_mape, smape_champ]])
resumen

**📖 Cómo se lee.** Quedan registrados **θ₁ −0,4326, Θ₁ −0,5475, MAPE 2,88 %, Prophet 7,45 %, sMAPE campeón 4,09 %**, con AIC, Ljung-Box y ADF de apoyo. A partir de aquí, **las figuras se generan leyendo este Excel**, no los objetos en memoria (convención del curso).

## Transversal — Gráficas de resultados leyendo el Excel (Sección 6 del cuaderno)

Las figuras de resultados se generan **desde el Excel** (no desde los objetos en memoria), como fija la convención del curso.

**🔎 Qué hace este código.** Lee la hoja `descomposicion` y traza los cuatro paneles STL (observado / tendencia / estacionalidad / residuo) → `figuras/S11_descomposicion.png`.

In [ ]:
xl = pd.ExcelFile(XLSX)

# (1) Descomposición STL
d = xl.parse("descomposicion", parse_dates=["fecha"])
fig, ax = plt.subplots(4, 1, figsize=(11, 8), sharex=True)
for a, col, tit in zip(ax, ["observado", "trend", "seasonal", "resid"],
                       ["Observado", "Tendencia", "Estacionalidad", "Residuo"]):
    a.plot(d["fecha"], d[col], color="#c8102e"); a.set_ylabel(tit)
ax[0].set_title("Descomposición STL de AirPassengers (desde Excel)")
ax[-1].set_xlabel("año")
plt.tight_layout(); fig.savefig(FIG / "S11_descomposicion.png", dpi=150, bbox_inches="tight"); plt.show()

**🔎 Qué hace este código.** Lee `pronostico_holdout` y grafica el **pronóstico del holdout 1960** (real vs. predicho) con el **intervalo del 95 %** → `figuras/S11_pronostico_holdout.png`.

In [ ]:
# (2) Pronóstico holdout con intervalo del 95 %
h = xl.parse("pronostico_holdout", parse_dates=["fecha"])
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(h["fecha"], h["real"], marker="o", color="#111111", label="real")
ax.plot(h["fecha"], h["pred"], marker="s", color="#c8102e", label="pronóstico SARIMA")
ax.fill_between(h["fecha"], h["lower"], h["upper"], color="#c8102e", alpha=0.18,
                label="intervalo 95 %")
ax.set_title("Modelo aerolínea — pronóstico holdout 1960 (desde Excel)")
ax.set_xlabel("mes"); ax.set_ylabel("pasajeros (miles)"); ax.legend()
plt.tight_layout(); fig.savefig(FIG / "S11_pronostico_holdout.png", dpi=150, bbox_inches="tight"); plt.show()

**🔎 Qué hace este código.** Lee `backtesting_store` y traza el **sMAPE medio por modelo** (barras: seasonal-naive, ETS, SARIMA) → `figuras/S11_backtesting_smape.png`. El campeón (menor sMAPE) va en rojo.

In [ ]:
# (3) Comparación de modelos por sMAPE medio (backtesting rolling-origin)
b = xl.parse("backtesting_store")
med = b[["snaive", "ets", "sarima"]].mean()
fig, ax = plt.subplots(figsize=(7, 4))
barras = ax.bar(["seasonal-naive", "ETS", "SARIMA"], med.values,
                color=["#9aa0a6", "#f4a6ae", "#c8102e"])
for r, v in zip(barras, med.values):
    ax.text(r.get_x() + r.get_width() / 2, v + 0.05, f"{v:.2f}%", ha="center", fontsize=9)
ax.set_title("Store Demand — sMAPE medio por modelo (rolling-origin, desde Excel)")
ax.set_ylabel("sMAPE (%)")
plt.tight_layout(); fig.savefig(FIG / "S11_backtesting_smape.png", dpi=150, bbox_inches="tight"); plt.show()
print("El campeón (menor sMAPE) debe batir al seasonal-naive para justificar su uso.")

**🔎 Qué hace este código.** Si Prophet corrió, lee `prophet_cv` y grafica el **MAPE por horizonte** → `figuras/S11_prophet_mape.png` (muestra que el error crece con el horizonte).

In [ ]:
# (4) MAPE de Prophet por horizonte (si Prophet corrió)
if "prophet_cv" in xl.sheet_names:
    p = xl.parse("prophet_cv").sort_values("horizonte_dias")
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(p["horizonte_dias"], p["mape_pct"], color="#c8102e")
    ax.set_title("Prophet — MAPE de cross_validation por horizonte (desde Excel)")
    ax.set_xlabel("horizonte (días)"); ax.set_ylabel("MAPE (%)")
    plt.tight_layout(); fig.savefig(FIG / "S11_prophet_mape.png", dpi=150, bbox_inches="tight"); plt.show()
    print("El MAPE crece con el horizonte: se reporta al horizonte que importa al negocio.")
else:
    print("Prophet no corrió en este entorno; se omite la gráfica de su backtesting (target opcional).")

## 11.7 — ¿Cómo se verifica que el resultado es real? ✅ Verificación desde la base (Sección 6.1 del cuaderno)

El Excel de contrato ya está escrito. Ahora se comprueba que **es producto de ejecutar el código sobre los datos**, no un registro independiente del cálculo: se **recomputa** lo clave —coeficientes θ₁/Θ₁, MAPE del holdout y Ljung-Box— con un ajuste **fresco** sobre `log(AirPassengers)` y se **cruza con el Excel** mediante `assert`. Refleja lo que hace el material de referencia de la sesión (que recomputa desde la base). **No toca el Excel.**

**🔎 Qué hace este código.** (1) Reajusta el modelo aerolínea sobre `log(s)` y extrae **θ₁**, **Θ₁** y el **Ljung-Box(12)**; (2) reajusta con el corte **132/12** y recomputa el **MAPE holdout**; y **cruza** los cuatro con la hoja `series_airpassengers` mediante `assert`.

In [ ]:
# ✅ VERIFICACIÓN DESDE LA BASE: recomputa coeficientes, MAPE y Ljung-Box y los cruza con el Excel (assert). NO toca el Excel.
meta_v = pd.read_excel(XLSX, sheet_name="series_airpassengers")
val = lambda k: float(meta_v.loc[meta_v["metrica"] == k, "valor"].iloc[0])

# (1) coeficientes del modelo aerolínea con un ajuste FRESCO sobre log(AirPassengers)
res_v = SARIMAX(np.log(s), order=(0, 1, 1), seasonal_order=(0, 1, 1, 12), trend="n",
                enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
ma1_re  = float(res_v.params["ma.L1"])
sma1_re = float(res_v.params["ma.S.L12"])
lb_re   = float(acorr_ljungbox(res_v.resid.iloc[13:], lags=[12], return_df=True)["lb_pvalue"].iloc[-1])
# (2) MAPE holdout recomputado con el corte 132/12
res_hv = SARIMAX(np.log(s.iloc[:-12]), order=(0, 1, 1), seasonal_order=(0, 1, 1, 12), trend="n",
                 enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
pred_v  = np.exp(res_hv.forecast(steps=12).to_numpy())
real_v  = s.iloc[-12:].to_numpy().astype(float)
mape_re = float(np.mean(np.abs((real_v - pred_v) / real_v)) * 100.0)

tabla_v = pd.DataFrame(
    [["ma.L1 (theta1)",       round(ma1_re, 4),  val("ma.L1 (theta1) modelo aerolinea"),      "-0,4326"],
     ["ma.S.L12 (Theta1)",    round(sma1_re, 4), val("ma.S.L12 (Theta1) estacional"),         "-0,5475"],
     ["MAPE holdout 12m (%)", round(mape_re, 4), val("MAPE holdout 12m AirPassengers (%)"),   "2,8836"],
     ["Ljung-Box(12) p",      round(lb_re, 4),   val("Ljung-Box(12) p-valor residuos"),       "0,7543"]],
    columns=["magnitud", "recomputado", "Excel", "target"])
display(tabla_v)

assert abs(ma1_re  - val("ma.L1 (theta1) modelo aerolinea"))       < 1e-3
assert abs(sma1_re - val("ma.S.L12 (Theta1) estacional"))          < 1e-3
assert abs(mape_re - val("MAPE holdout 12m AirPassengers (%)"))    < 1e-2
assert abs(lb_re   - val("Ljung-Box(12) p-valor residuos"))        < 1e-2
print("assert OK: lo recomputado desde la base coincide con el Excel de contrato (no es un registro suelto).")

**📖 Cómo se lee.** Las cuatro magnitudes recomputadas desde los datos —**θ₁ −0,4326, Θ₁ −0,5475, MAPE 2,8836 %, Ljung-Box 0,7543**— coinciden con la hoja `series_airpassengers`: el Excel **es** producto de la ejecución, no un valor tecleado. Es la contraparte en el cuaderno de el material de referencia de la sesión.

## 11.3 en profundidad — Construcción desde cero (descomposición del alumno) (Sección 7 del cuaderno)

Para probar que se entiende el **flujo de pronóstico** —no solo la llamada a la librería— se rearma el pipeline completo **sin los helpers** y se reproduce el contrato:
- **🧱 Flujo de pronóstico:** `log` → **diferenciar `(1−B)(1−B¹²)`** (comprobar estacionariedad) → **ajustar SARIMA(0,1,1)(0,1,1)₁₂** sobre el train → **pronosticar el holdout** → **des-transformar con `exp`** → **MAPE**. Reproduce el **MAPE 2,88 %** del contrato.

**🔎 Qué hace este código.** Ejecuta ese flujo paso a paso sobre AirPassengers (transforma, diferencia y prueba ADF de forma manual, ajusta, pronostica 12 meses, des-transforma y calcula el MAPE) y **verifica con `assert`** que reproduce el MAPE de la hoja `series_airpassengers`.

In [ ]:
# 🧱 CONSTRUYE DESDE CERO: el flujo de pronóstico sin helpers (log -> diferenciar -> SARIMA -> holdout -> MAPE). NO toca el Excel.
serie = s.astype(float)
log_serie = np.log(serie)                                   # 1) transformar (serie multiplicativa)

z = np.log(serie.values)                                    # 2) diferenciar (1-B)(1-B^12) a mano ...
z = z[1:] - z[:-1]; z = z[12:] - z[:-12]
adf_p_cero = float(adfuller(z, autolag="AIC")[1])           #    ... y confirmar estacionariedad

tr = log_serie.iloc[:-12]                                   # 3) holdout = últimos 12 meses
mod_cero = SARIMAX(tr, order=(0, 1, 1), seasonal_order=(0, 1, 1, 12), trend="n",
                   enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
fc_cero   = np.exp(mod_cero.forecast(steps=12).to_numpy())  # 4) pronosticar y des-transformar con exp
real_cero = serie.iloc[-12:].to_numpy()
mape_cero = float(np.mean(np.abs((real_cero - fc_cero) / real_cero)) * 100.0)   # 5) MAPE en pasajeros

print(f"ADF (serie diferenciada a mano) p = {adf_p_cero:.4f}  -> estacionaria, justifica d=1, D=1")
print(f"MAPE holdout del pipeline reconstruido = {mape_cero:.4f} %")

meta_c  = pd.read_excel(XLSX, sheet_name="series_airpassengers")
mape_xl = float(meta_c.loc[meta_c["metrica"] == "MAPE holdout 12m AirPassengers (%)", "valor"].iloc[0])
assert abs(mape_cero - mape_xl) < 1e-6, "el pipeline reconstruido debe reproducir el MAPE del contrato"
print(f"\nassert OK: el flujo reconstruido desde cero reproduce el contrato (MAPE {mape_cero:.4f}% == Excel {mape_xl:.4f}%).")

**📖 Cómo se lee.** El pipeline rearmado manualmente —transformar, diferenciar, ajustar el orden identificado, pronosticar y des-transformar— reproduce **exactamente** el **MAPE 2,88 %** del contrato. La lección: el resultado no es una caja negra, sino la cadena `log → (1−B)(1−B¹²) → SARIMA → exp → MAPE` (la ficha de la sesión de réplica del paper, punto 5).

## 11.6 — ¿Cuándo se puede confiar en este pronóstico? Supuestos: decisiones y condiciones de validez (Sección 8 del cuaderno)

Una serie temporal no es una muestra de filas intercambiables: sus observaciones están **ordenadas y correlacionadas**. Los «supuestos» de esta sesión son de dos tipos: **(a) supuestos del modelo ARIMA/SARIMA** (estacionariedad, identificación del orden, ruido blanco de los residuos, forma aditiva/multiplicativa) y **(b) decisiones metodológicas** de validación temporal (no barajar; backtesting contra una línea base). La **fuente canónica** es la guía de supuestos de la sesión; aquí se ejecutan **seis diagnósticos** que la ilustran. **Ninguno escribe en el Excel de contrato.**

| Diagnóstico | Qué revela | Fuente (`SUPUESTOS_S11.md`) |
|---|---|---|
| 8.1 Estacionariedad (ADF/KPSS) | La log-serie en nivel es NO estacionaria; tras `(1−B)(1−B¹²)` sí lo es | punto 2.1 y 2.3 |
| 8.2 Identificación del orden (ACF/PACF) | Picos en lag 1 y lag 12 → `(0,1,1)(0,1,1)[12]` | punto 2.4 |
| 8.3 Ruido blanco (Ljung-Box) | Los residuos del aerolínea no tienen autocorrelación (p alto) | punto 2.5 |
| 8.4 Aditiva vs. multiplicativa | La varianza estacional crece con el nivel → log | punto 2.2 |
| 8.5 Fuga temporal (leakage) | Barajar el split subestima el error (métrica «demasiado buena») | punto 4.1 |
| 8.6 Backtesting rolling-origin | El campeón debe superar al seasonal-naive | punto 4.2 |

**🔎 Qué hace este código (Diagnóstico 8.1 — estacionariedad).** Corre **ADF** y **KPSS** sobre la log-serie **en nivel** y **tras diferenciar** `(1−B)(1−B¹²)`, y verifica con `assert` la lectura de la matriz (nivel no estacionario; diferenciada estacionaria). No escribe en el Excel.

In [ ]:
# Diagnóstico 8.1 (estacionariedad): ADF y KPSS ANTES y DESPUÉS de diferenciar — NO escribe en el Excel
from statsmodels.tsa.stattools import adfuller, kpss
def _pruebas(x):
    adf_p = float(adfuller(x, autolag="AIC")[1])
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        kpss_p = float(kpss(x, regression="c", nlags="auto")[1])
    return adf_p, kpss_p

niveles = np.log(s)
dd = niveles.diff().dropna().diff(12).dropna()
res_niv = _pruebas(niveles); res_dif = _pruebas(dd)
print(f"{'log(y) (nivel)':32s}  ADF p={res_niv[0]:6.4f}  KPSS p={res_niv[1]:6.4f}")
print(f"{'log(y) dif. reg.+estacional':32s}  ADF p={res_dif[0]:6.4f}  KPSS p={res_dif[1]:6.4f}")
print("\nLectura: en NIVEL ADF NO rechaza (no estacionaria) y KPSS rechaza; tras DIFERENCIAR se invierten")
print("-> estacionaria. Ver la guía de supuestos de la sesión, punto 2.1 (matriz ADF x KPSS).")
assert res_niv[0] > 0.05 and res_dif[0] < 0.05, "nivel no estacionario y diferenciada estacionaria (ADF)"

**📖 Cómo se lee.** En **nivel** el ADF **no rechaza** (no estacionaria) y el KPSS **rechaza**; **tras diferenciar** los veredictos se **invierten** hacia estacionariedad. Es la matriz ADF×KPSS en acción: como sus nulas son opuestas, se confirman entre sí (la guía de supuestos de la sesión, punto 2.1).

**🔎 Qué hace este código (Diagnóstico 8.2 — identificación del orden).** Calcula la **ACF** y la **PACF** numéricas de la doble diferencia y compara los lags 1 y 12 contra la banda de significancia ±1,96/√n para justificar `(0,1,1)(0,1,1)[12]`. No escribe en el Excel.

In [ ]:
# Diagnóstico 8.2 (identificación del orden): ACF/PACF de la doble diferencia — NO escribe en el Excel
from statsmodels.tsa.stattools import acf, pacf
dd = np.log(s).diff().dropna().diff(12).dropna()
r  = acf(dd, nlags=13)
pr = pacf(dd, nlags=13, method="ywm")
banda = 1.96 / np.sqrt(len(dd))
print(f"Banda de significancia +/-{banda:.3f}  (n={len(dd)})")
print(f"ACF  lag 1  = {r[1]:+.3f}   ACF  lag 12 = {r[12]:+.3f}   (fuera de banda -> MA regular q=1 y MA estacional Q=1)")
print(f"PACF lag 1  = {pr[1]:+.3f}  PACF lag 12 = {pr[12]:+.3f}")
print("Lectura: picos en lag 1 y lag 12 de la ACF => (0,1,1)(0,1,1)[12]. Ver SUPUESTOS_S11.md, punto 2.4.")
assert abs(r[12]) > banda, "el lag estacional 12 debe sobresalir de la banda (justifica Q=1)"

**📖 Cómo se lee.** La **ACF sobresale en el lag 1 y en el lag 12** (fuera de la banda), la firma de un **MA regular** y un **MA estacional**: exactamente el orden `(0,1,1)(0,1,1)[12]` del modelo aerolínea. ⚠️ La regla es ACF↔MA, PACF↔AR; confundirlas lleva a órdenes equivocados (la guía de supuestos de la sesión, punto 2.4).

**❓ Qué se quiere averiguar.** ¿Queda en los residuos algún patrón que el modelo no supo capturar? Y si queda, ¿en qué se nota para quien usa el pronóstico?

- **Qué decide:** si el intervalo del 95 % que se entrega a planificación es creíble. Un residuo con estructura pendiente significa que el modelo **subestima su propia incertidumbre**: la banda sale más estrecha de lo que corresponde y el stock de seguridad se calcula corto.
- **Antes de mirar el resultado:** la nula de Ljung-Box es «no hay autocorrelación». Si algún rezago (6, 12, 18 o 24) diera **p < 0,05**, quedaría estructura sin modelar y el ciclo de Box-Jenkins tendría que volver a la identificación del orden. Si todos superan 0,05 —en especial el **rezago 12**, el estacional—, los residuos se comportan como ruido blanco y el ajuste puede darse por cerrado.

**🔎 Qué hace este código (Diagnóstico 8.3 — ruido blanco).** Corre **Ljung-Box** sobre los residuos del modelo aerolínea a varios rezagos (6, 12, 18, 24) y verifica con `assert` que el p-valor a 12 supera 0,05 (residuos ≈ ruido blanco). No escribe en el Excel.

In [ ]:
# Diagnóstico 8.3 (ruido blanco): Ljung-Box de los residuos del modelo aerolínea — NO escribe en el Excel
lb = acorr_ljungbox(res.resid.iloc[13:], lags=[6, 12, 18, 24], return_df=True)
print(lb.round(4))
p12 = float(lb.loc[12, "lb_pvalue"])
print(f"\nLjung-Box(12) p = {p12:.3f}  (> 0,05 -> residuos aprox. ruido blanco: el modelo capturó la estructura)")
print("Lectura: aquí un p ALTO es lo DESEABLE (invertir esa dirección es el error frecuente). Ver SUPUESTOS_S11.md, punto 2.5.")
assert p12 > 0.05, "los residuos del modelo aerolínea deben ser ruido blanco (Ljung-Box p > 0,05)"

**📖 Cómo se lee.** Ningún rezago rechaza la nula de **no autocorrelación**: los residuos son **ruido blanco** (p₁₂ = 0,754), criterio de salida del ciclo Box-Jenkins. Si el p fuera bajo, quedaría estructura sin capturar y los **intervalos serían demasiado estrechos** (la guía de supuestos de la sesión, punto 2.5).

**🔎 Qué hace este código (Diagnóstico 8.4 — aditiva vs. multiplicativa).** Descompone la serie con un modelo **aditivo** y mide la **varianza del residuo** en la primera vs. la segunda mitad: si crece, la estacionalidad es **multiplicativa** y conviene el `log`. Verifica con `assert`. No escribe en el Excel.

In [ ]:
# Diagnóstico 8.4 (aditiva vs. multiplicativa): varianza del residuo aditivo a lo largo del tiempo — NO escribe en el Excel
from statsmodels.tsa.seasonal import seasonal_decompose
add = seasonal_decompose(s, model="additive", period=12)
resid_add = add.resid.dropna()
mitad = len(resid_add) // 2
var_ini = float(resid_add.iloc[:mitad].var())
var_fin = float(resid_add.iloc[mitad:].var())
print(f"Residuo del modelo ADITIVO: var 1a mitad = {var_ini:.1f}  |  var 2a mitad = {var_fin:.1f}")
print(f"Cociente var_fin/var_ini = {var_fin / var_ini:.2f}  (>1 -> amplitud crece con el nivel -> MULTIPLICATIVA -> log)")
print("Lectura: la varianza del residuo aditivo crece con el tiempo; el log la estabiliza. Ver SUPUESTOS_S11.md, punto 2.2.")
assert var_fin > var_ini, "en AirPassengers la varianza estacional crece con el nivel (multiplicativa)"

**📖 Cómo se lee.** Con un modelo **aditivo**, la varianza del residuo es **mayor en la segunda mitad**: el patrón estacional se agranda con el nivel → la serie es **multiplicativa** y por eso el modelo aerolínea trabaja sobre `log(y)` (la guía de supuestos de la sesión, punto 2.2).

**❓ Qué se quiere averiguar.** ¿Qué ocurre si una serie se valida como se valida cualquier tabla, con un **K-fold aleatorio**? ¿Cuánto se equivoca el número que se reporta?

- **Qué decide:** si la cifra que se le promete al negocio es la que se verá en producción. Es la diferencia entre un modelo que cumple y uno que se cae el primer mes, sin que nadie entienda por qué la validación indicaba un resultado distinto.
- **Antes de mirar el resultado:** al barajar meses, el modelo se entrena con enero y julio de 2017 para predecir abril de ese mismo año, es decir, con información que **en el momento de decidir no existía**. Si el RMSE del K-fold saliera igual o mayor que el de la validación de origen móvil, barajar sería inofensivo y la advertencia sobraría. Si resulta **menor**, la diferencia mide exactamente el sesgo optimista: cuánto mejor parece el modelo por haber visto el futuro. Una brecha del orden del 20 % basta para aprobar un proyecto que no debía aprobarse.

**🔎 Qué hace este código (Diagnóstico 8.5 — fuga temporal).** Construye una matriz de **12 rezagos** de la serie mensual de negocio y compara el RMSE de una regresión lineal con **validación que respeta el tiempo** (`TimeSeriesSplit`) frente a un **K-fold ALEATORIO** (baraja pasado y futuro). Muestra que barajar **subestima** el error. No escribe en el Excel.

In [ ]:
# Diagnóstico 8.5 (fuga temporal): barajar el split subestima el error — NO escribe en el Excel
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, TimeSeriesSplit
from sklearn.metrics import mean_squared_error

vals = monthly.values.astype(float)
L = 12                                     # predecir el mes t con los 12 meses previos (rezagos)
Xlag = np.column_stack([vals[i:len(vals) - L + i] for i in range(L)])
ylag = vals[L:]

def _rmse_cv(splitter):
    errs = []
    for tr, te in splitter.split(Xlag):
        pr = LinearRegression().fit(Xlag[tr], ylag[tr]).predict(Xlag[te])
        errs.append(mean_squared_error(ylag[te], pr) ** 0.5)
    return float(np.mean(errs))

rmse_time = _rmse_cv(TimeSeriesSplit(n_splits=5))            # HONESTO: respeta el orden temporal
rmse_shuf = _rmse_cv(KFold(n_splits=5, shuffle=True, random_state=0))   # CON FUGA: baraja
print(f"RMSE con validación temporal (honesta, origen móvil) = {rmse_time:.2f}")
print(f"RMSE con K-fold ALEATORIO (fuga temporal)            = {rmse_shuf:.2f}   <-- se ve MEJOR por barajar")
print(f"El barajado subestima el error en {100 * (1 - rmse_shuf / rmse_time):.0f} %.")
print("Lectura: en series NUNCA se baraja; se valida con origen móvil. Ver SUPUESTOS_S11.md, punto 4.1.")
assert rmse_shuf < rmse_time, "el K-fold aleatorio subestima el error (fuga temporal)"

**📖 Cómo se lee.** El **K-fold aleatorio** reporta un RMSE **menor** que la validación temporal: barajar deja que el modelo «vea» meses vecinos del futuro, y la métrica sale **demasiado buena**. En producción ese modelo **falla**. Por eso las series se validan con **origen móvil**, nunca con K-fold aleatorio (la guía de supuestos de la sesión, punto 4.1; Albelali & Ahmed 2025).

**🔎 Qué hace este código (Diagnóstico 8.6 — backtesting rolling-origin).** Lee del Excel el **sMAPE medio por modelo** y verifica con `assert` que el **campeón (SARIMA)** supera al **seasonal-naive**. No escribe en el Excel.

In [ ]:
# Diagnóstico 8.6 (backtesting): el campeón debe batir al seasonal-naive — NO escribe en el Excel
b = pd.read_excel(XLSX, sheet_name="backtesting_store")
med = b[["snaive", "ets", "sarima"]].mean()
print("sMAPE medio por modelo (rolling-origin, leído del Excel):")
for k in ["snaive", "ets", "sarima"]:
    print(f"  {k:8s} = {med[k]:.2f} %")
print(f"\nCampeón = SARIMA ({med['sarima']:.2f} %) < ETS ({med['ets']:.2f} %) < seasonal-naive ({med['snaive']:.2f} %).")
print("Lectura: un modelo que no bate al naive estacional NO aporta. Ver SUPUESTOS_S11.md, punto 4.2.")
assert med["sarima"] <= med["snaive"], "el campeón (SARIMA) debe batir al seasonal-naive"

**📖 Cómo se lee.** El **SARIMA (4,09 %)** queda por debajo del **ETS (5,50 %)** y del **seasonal-naive (5,78 %)**: es el **campeón** y **justifica su uso** por superar a la línea base obligatoria. Sin esa comparación no se sabría si el modelo aporta (la guía de supuestos de la sesión, punto 4.2).

## Práctica — Drills (enunciados) (Sección 9 del cuaderno)

Los enunciados completos y la rúbrica están en `evaluacion/drills.docx`. En parejas; entregable individual.

1. **Identificar (p,d,q) a partir de ACF/PACF.** Tomar la log-serie de AirPassengers doblemente diferenciada (paso 4) y, leyendo los correlogramas, **proponer y justificar** el orden estacional y no estacional. Comparar la propuesta con el orden del modelo aerolínea.
2. **Holt-Winters aditivo vs. multiplicativo.** Ajustar ambos sobre AirPassengers y sobre la serie mensual de Store Demand; decidir cuál corresponde según cómo escala la estacionalidad y **argumentar la elección** con el AIC y el patrón visual.
3. **Rolling-origin con tres modelos.** Repetir el backtesting de origen móvil (paso 11) para **otra serie** (p. ej. tienda 1, producto 5): comparar seasonal-naive, ETS y SARIMA por sMAPE, elegir el campeón, verificar que **supera al naive** y documentar que no hay fuga temporal.

## 11.9 — ¿Qué no se puede afirmar, y qué sigue en S12? Cierre (Sección 10 del cuaderno)

**Entregable evaluable** (`evaluacion/entregable.docx`, rúbrica vigesimal 0–20; plantilla `plantillas/plantilla_pronostico_backtesting.docx`). Construir un **pronóstico mensual de demanda** de un producto de Store Item Demand: descomponer, evaluar estacionariedad, ajustar y comparar **≥3 modelos** (SARIMA/ETS/Prophet o ML) con **intervalos de predicción**, validar con **rolling-origin** (sMAPE/RMSE, sin fuga temporal), elegir el campeón y comunicar la recomendación.

**Control corto de cierre.** Habrá un control breve sobre: matriz ADF/KPSS, lectura de ACF/PACF, convención de signos MA, elección aditivo/multiplicativo, y detección de fuga temporal.

**Vínculo con el proyecto integrador.** La descomposición y la estacionariedad de esta sesión alimentan la fase de modelado del proyecto: todo pronóstico se valida con **origen móvil** y se compara contra la línea base **seasonal-naive**.

### Para seguir explorando (fuentes de actualidad, las fuentes de actualidad de la sesión)
- **Hidden Leaks in Time Series Forecasting** (Albelali & Ahmed, arXiv 2512.06932, 07/12/2025): construir las secuencias antes del split da una validación optimista; el error honesto es ~20 % mayor que el que reporta esa validación con fuga.
- **Rethinking Evaluation in the Era of Time Series Foundation Models** (Meyer et al., arXiv 2510.13654, 2025): fuga por solapamiento train-test y temporal en modelos fundacionales.
- **On the retraining frequency of global models in retail demand forecasting** (Zanotti, arXiv 2505.00356, 2025): reentrenar con menor frecuencia mantiene la precisión y reduce el costo.

**Lecturas base.** FPP3 caps. 3, 8 y 9 (Hyndman & Athanasopoulos); Box & Jenkins (1970); Hyndman & Khandakar (2008); Taylor & Letham (2018); Makridakis et al. (2020), M4. Detalle en la bibliografía de la sesión.